Nama: Ni Made Era Purnama Dewi
NIM: 2501010007
Kelas-E

In [ ]:
import os, json, hashlib, datetime as dt
import pandas as pd

CAMPUS_EMAIL = "info@instiki.ac.id"

# Parameter akses web (dipakai nanti di Soal 3 dan 4)
USER_AGENT = f"INSTIKI-UTS-DAM/1.0 (mahasiswa 2501010007; kontak: {CAMPUS_EMAIL})"
TIMEOUT = 20      # detik, wajib di setiap request
DELAY = 2.0       # jeda antar-request (detik)

# Struktur folder
RAW, PROC, META = "data/raw", "data/processed", "data/meta"
for p in (RAW, PROC, META):
    os.makedirs(p, exist_ok=True)

# Fungsi hash MD5 (untuk provenance dan bukti berkas tidak berubah)
def md5_file(path):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            h.update(chunk)
    return h.hexdigest()

print("User-Agent:", USER_AGENT)

User-Agent: INSTIKI-UTS-DAM/1.0 (mahasiswa 2501010007; kontak: info@instiki.ac.id)


In [ ]:
import shutil
from google.colab import files

up = files.upload()                      # akan muncul tombol "Choose Files"
fn = list(up.keys())[0]                  # nama berkas yang diunggah
RAW_CSV = os.path.join(RAW, fn)
shutil.move(fn, RAW_CSV)                 # pindahkan ke data/raw
os.chmod(RAW_CSV, 0o444)                 # kunci read-only (bukti raw tidak diubah)

MD5_AWAL = md5_file(RAW_CSV)
print("Berkas:", RAW_CSV)
print("MD5   :", MD5_AWAL)

Saving data_20261115155634-daftar-destinasi-wisata-berdasarkan-kawasan-strategis-pariwisata-daerah-kspd-provinsi-nusa-tenggara-barat.csv to data_20261115155634-daftar-destinasi-wisata-berdasarkan-kawasan-strategis-pariwisata-daerah-kspd-provinsi-nusa-tenggara-barat.csv
Berkas: data/raw/data_20261115155634-daftar-destinasi-wisata-berdasarkan-kawasan-strategis-pariwisata-daerah-kspd-provinsi-nusa-tenggara-barat.csv
MD5   : 358dd8b2be8b073b32849f85da7f632a


In [ ]:
# Isi dari halaman dataset di portal sumber (buka halamannya)
SOURCE_URL  = "isi_url_halaman_dataset"
LICENSE_TXT = "isi_lisensi_sesuai_halaman_dataset"

PROVENANCE = []   # daftar catatan semua sumber (dipakai sampai Soal 5)

def add_provenance(nama, url, path, lisensi, metode):
    PROVENANCE.append({
        "sumber": nama, "url": url,
        "tanggal_akses": dt.datetime.now().isoformat(timespec="seconds"),
        "lisensi": lisensi, "metode": metode, "berkas": path,
        "ukuran_bytes": os.path.getsize(path), "md5": md5_file(path)})

add_provenance("Dataset destinasi wisata KSPD NTB (CSV)", SOURCE_URL, RAW_CSV, LICENSE_TXT, "unduh berkas")
pd.DataFrame(PROVENANCE)

,sumber,url,tanggal_akses,lisensi,metode,berkas,ukuran_bytes,md5
0,Dataset destinasi wisata KSPD NTB (CSV),isi_url_halaman_dataset,2026-11-16T02:27:50,isi_lisensi_sesuai_halaman_dataset,unduh berkas,data/raw/data_20261115155634-daftar-destinasi-...,73734,358dd8b2be8b073b32849f85da7f632a


In [ ]:
# Hasil inspeksi berkas: pemisah koma, encoding UTF-8, header di baris pertama,
# semua field memakai tanda kutip.
# dtype dibuat eksplisit; Kode Provinsi = string karena itu kode, bukan angka untuk dihitung.
df = pd.read_csv(
    RAW_CSV,
    sep=",",
    encoding="utf-8",
    skiprows=0,
    header=0,
    dtype={"#": "int64", "Kode Provinsi": "string", "Nama Provinsi": "string",
           "KSPD": "string", "Destinasi": "string", "Uraian": "string",
           "Koordinat": "string", "Tahun": "int64"},
    na_values=["", "-", "NA", "N/A", "null"]
)

print("Shape:", df.shape)
df.head(3)

Shape: (266, 8)


,#,Kode Provinsi,Nama Provinsi,KSPD,Destinasi,Uraian,Koordinat,Tahun
0,1,52,Nusa Tenggara Barat,KSPD WAWORADA - SAPE DAN SEKITARNYA,Sigi Kamina/kalo Du,• Tipologi : Wisata Ziarah-religi • Lokasi : D...,"-8.665510, 118.845924",2025
1,2,52,Nusa Tenggara Barat,KSPD WAWORADA - SAPE DAN SEKITARNYA,Kompleks Rumah Adat Wawo,• Tipologi : Wisata Budaya • Lokasi : Desa Mar...,"-8.524548, 118.846067",2025
2,3,52,Nusa Tenggara Barat,KSPD WAWORADA - SAPE DAN SEKITARNYA,Desa Adat Sambori,• Tipologi : Wisata Alam Dan Wisata Budaya • L...,"-8.606385, 118.778023",2025


In [ ]:
# IDENTIFIKASI DATASET
size_b = os.path.getsize(RAW_CSV)

print("Jenis data    : TERSTRUKTUR (tabel baris-kolom)")
print("                kolom 'Uraian' berisi teks semi-terstruktur berpola '• Tipologi : ... • Lokasi : ...'")
print(f"Baris x kolom : {df.shape[0]} x {df.shape[1]}  (minimal 100 baris: {'OK' if df.shape[0] >= 100 else 'TIDAK'})")
print(f"Ukuran berkas : {size_b:,} bytes = {size_b/1024:.1f} KB")

# Tipe data, jumlah kosong, dan jumlah nilai unik tiap kolom
print("\nTipe data & kelengkapan:")
display(pd.DataFrame({"dtype": df.dtypes.astype(str), "null": df.isna().sum(), "unik": df.nunique()}))

# Pecah koordinat menjadi lat dan lon (satuan: derajat desimal)
ll = df["Koordinat"].str.split(",", expand=True).astype(float)
df["lat"], df["lon"] = ll[0], ll[1]

print("\nSatuan  : koordinat = derajat desimal (lat, lon); jarak di 'Uraian' = Km; waktu tempuh = Jam/Menit")
print("Wilayah :", df["Nama Provinsi"].unique().tolist(), "|", df["KSPD"].nunique(), "KSPD")
print(f"Bbox    : lat {df.lat.min():.3f} s/d {df.lat.max():.3f}, lon {df.lon.min():.3f} s/d {df.lon.max():.3f}")
print("Waktu   :", sorted(df["Tahun"].unique().tolist()), "(satu tahun, tanpa kolom tanggal)")

print("\nJumlah destinasi per KSPD:")
display(df["KSPD"].value_counts().to_frame("jumlah destinasi"))

Jenis data    : TERSTRUKTUR (tabel baris-kolom)
                kolom 'Uraian' berisi teks semi-terstruktur berpola '• Tipologi : ... • Lokasi : ...'
Baris x kolom : 266 x 8  (minimal 100 baris: OK)
Ukuran berkas : 73,734 bytes = 72.0 KB

Tipe data & kelengkapan:


,dtype,null,unik
#,int64,0,266
Kode Provinsi,string,0,1
Nama Provinsi,string,0,1
KSPD,string,0,11
Destinasi,string,0,266
Uraian,string,0,254
Koordinat,string,0,264
Tahun,int64,0,1



Satuan  : koordinat = derajat desimal (lat, lon); jarak di 'Uraian' = Km; waktu tempuh = Jam/Menit
Wilayah : ['Nusa Tenggara Barat'] | 11 KSPD
Bbox    : lat -9.041 s/d -8.110, lon 115.861 s/d 119.226
Waktu   : [2025] (satu tahun, tanpa kolom tanggal)

Jumlah destinasi per KSPD:


,jumlah destinasi
KSPD,
KSPD KUTA - MANDALIKA DAN SEKITARNYA,54
"KSPD SENGGIGI, 3 GILI DAN SEKITARNYA",36
KSPD RASIMAS - SEMBALUN DAN SEKITARNYA,30
KSPD POTOTANO - MALUK DAN SEKITARNYA,28
KSPD SAMOTA (TELUK SALEH - MOYO – TAMBORA) DAN SEKITARNYA,26
KSPD MATARAM METRO DAN SEKITARNYA,26
KSPD WAWORADA - SAPE DAN SEKITARNYA,18
KSPD ALAS UTAN DAN SEKITARNYA,15
KSPD TELUK BIMA DAN SEKITARNYA,15


In [ ]:
# ESTIMASI STORAGE 1 TAHUN
# Asumsi:
#  (1) satu snapshot berkas dengan ukuran sama disimpan setiap bulan (tanpa deduplikasi)
#  (2) cadangan 30% untuk versi hasil olahan, log, dan backup
BULAN, CADANGAN = 12, 0.30

estimasi = size_b * BULAN * (1 + CADANGAN)
print(f"Ukuran berkas : {size_b:,} B")
print(f"Rumus         : {size_b:,} B x {BULAN} bulan x (1 + {CADANGAN})")
print(f"Estimasi      : {estimasi:,.0f} B = {estimasi/1024:.1f} KB = {estimasi/1024**2:.2f} MB")

Ukuran berkas : 73,734 B
Rumus         : 73,734 B x 12 bulan x (1 + 0.3)
Estimasi      : 1,150,250 B = 1123.3 KB = 1.10 MB


In [ ]:
import re

# Kolom turunan hanya di memori; berkas raw tidak disentuh.
df["tipologi"]      = df["Uraian"].str.extract(r"Tipologi\s*:\s*(.*?)\s*(?:•|$)")[0]
df["lokasi_txt"]    = df["Uraian"].str.extract(r"Lokasi\s*:\s*(.*?)\s*(?:•|$)")[0]
df["aksesibilitas"] = df["Uraian"].str.extract(r"Aksesibilitas\s*:\s*(.*)$")[0]

# Kunci kabupaten/kota, dicari dari teks Lokasi saja (KSB = Kab. Sumbawa Barat).
# Urutan penting: pola yang lebih spesifik harus dicek lebih dulu
# (mis. "Sumbawa Barat" sebelum "Sumbawa", "Kota Bima" sebelum "Bima").
KAB_PATTERNS = [
    ("Kota Mataram",  r"Kota Mataram|Mataram"),
    ("Kota Bima",     r"Kota Bima"),
    ("Lombok Utara",  r"Lombok Utara"),
    ("Lombok Barat",  r"Lombok Barat"),
    ("Lombok Tengah", r"Lombok Tengah"),
    ("Lombok Timur",  r"Lombok Timur"),
    ("Sumbawa Barat", r"Sumbawa Barat|\bKSB\b"),
    ("Sumbawa",       r"Sumbawa"),
    ("Dompu",         r"Dompu"),
    ("Bima",          r"Bima"),
]

def kab_key(teks):
    if not isinstance(teks, str):
        return None
    for key, pola in KAB_PATTERNS:
        if re.search(pola, teks, re.I):
            return key
    return None

df["kab_key"] = df["lokasi_txt"].map(kab_key)

print(df["kab_key"].value_counts(dropna=False))
df[["Destinasi", "tipologi", "lokasi_txt", "kab_key"]].head(5)

kab_key
None             43
Sumbawa          41
Lombok Timur     36
Lombok Tengah    28
Sumbawa Barat    27
Lombok Barat     24
Bima             23
Kota Mataram     20
Dompu            13
Lombok Utara      7
Kota Bima         4
Name: count, dtype: int64


,Destinasi,tipologi,lokasi_txt,kab_key
0,Sigi Kamina/kalo Du,Wisata Ziarah-religi,"Dusun Kamina, Desa Kalodu, Kec. Langgudu, Kab....",Bima
1,Kompleks Rumah Adat Wawo,Wisata Budaya,"Desa Maria Utara, Kec. Wawo, Kab. Bima",Bima
2,Desa Adat Sambori,Wisata Alam Dan Wisata Budaya,"Desa Sambori, Kec. Lambitu, Kab. Bima",Bima
3,Wisata Alam Kalate Ndira,Wisata Alam,"Dusun ‘diha, Desa Sie, Kec. Monta, Kab. Bima",Bima
4,Air Terjun Tarlawi Wawo,Wisata Alam,"Desa Tarlawi, Kec. Wawo, Kab. Bima",Bima


In [ ]:
# LISENSI & KETENTUAN PORTAL
# Semua isian diambil dari halaman sumber (dicek 15 Nov 2026), bukan dari asumsi.
SOURCE_URL  = "https://data.ntbprov.go.id/dataset/a2fbd8bd-6b85-4ce8-b743-a36c78e9f59d/show"
LICENSE_TXT = "License Not Specified (tidak ditentukan di halaman dataset; level akses: Public)"

LISENSI = {
    "portal"         : "Satu Data NTB (data.ntbprov.go.id)",
    "url_dataset"    : SOURCE_URL,
    "lisensi_dataset": LICENSE_TXT,
    "penyedia_data"  : "Dinas Pariwisata (sumber: Dinas Pariwisata dan Ekonomi Kreatif) - Provinsi NTB",
    "syarat_portal"  : ("Tautan 'Ketentuan Pengguna' di footer portal sudah dicoba dibuka (15 Nov 2026), "
                        "tetapi tidak menampilkan teks ketentuan (hanya muncul pop-up survei kepuasan). "
                        "Halaman dataset: lisensi 'License Not Specified', level akses Public. "
                        "Data dipakai hanya untuk keperluan akademik dengan menyebut sumber (Dinas Pariwisata, Satu Data NTB)."),
    "tanggal_cek"    : dt.date.today().isoformat(),
}

for k, v in LISENSI.items():
    print(f"{k:16}: {v}")

# Perbarui catatan provenance agar memakai isian ini
PROVENANCE[0]["url"] = SOURCE_URL
PROVENANCE[0]["lisensi"] = LICENSE_TXT

portal          : Satu Data NTB (data.ntbprov.go.id)
url_dataset     : https://data.ntbprov.go.id/dataset/a2fbd8bd-6b85-4ce8-b743-a36c78e9f59d/show
lisensi_dataset : License Not Specified (tidak ditentukan di halaman dataset; level akses: Public)
penyedia_data   : Dinas Pariwisata (sumber: Dinas Pariwisata dan Ekonomi Kreatif) - Provinsi NTB
syarat_portal   : Tautan 'Ketentuan Pengguna' di footer portal sudah dicoba dibuka (15 Nov 2026), tetapi tidak menampilkan teks ketentuan (hanya muncul pop-up survei kepuasan). Halaman dataset: lisensi 'License Not Specified', level akses Public. Data dipakai hanya untuk keperluan akademik dengan menyebut sumber (Dinas Pariwisata, Satu Data NTB).
tanggal_cek     : 2026-11-16


In [ ]:
# PEMINDAIAN PII DENGAN REGEX
PII_REGEX = {
    "email":   r"[\w.+-]+@[\w-]+\.[\w.-]+",
    "telepon": r"(?:\+62|62|0)8\d{8,11}",
    "NIK_16d": r"\b\d{16}\b",
    "medsos":  r"(?:instagram|facebook|tiktok|twitter)\.com",
}

hasil = []
for col in df.select_dtypes(include=["string", "object"]).columns:
    for nama, pola in PII_REGEX.items():
        n = df[col].astype(str).str.contains(pola, regex=True, na=False).sum()
        if n:
            hasil.append({"kolom": col, "pola": nama, "jumlah_baris": int(n)})

pii_hit = pd.DataFrame(hasil, columns=["kolom", "pola", "jumlah_baris"])
print("Jumlah temuan pola PII:", len(pii_hit))
display(pii_hit)

# KLASIFIKASI KOLOM
klasifikasi = pd.DataFrame([
    ["#, Kode/Nama Provinsi, Tahun", "Bukan PII (nomor urut dan atribut wilayah/waktu)"],
    ["KSPD, Destinasi, tipologi",    "Bukan PII (nama kawasan dan tempat wisata publik)"],
    ["Koordinat (lat, lon)",         "Quasi-identifier RENDAH: titik tempat wisata publik, bukan alamat individu"],
    ["Uraian (dusun/desa/kecamatan)","Quasi-identifier RENDAH: lokasi administratif sampai tingkat dusun"],
], columns=["kolom", "klasifikasi"])
display(klasifikasi)

Jumlah temuan pola PII: 0


,kolom,pola,jumlah_baris


,kolom,klasifikasi
0,"#, Kode/Nama Provinsi, Tahun",Bukan PII (nomor urut dan atribut wilayah/waktu)
1,"KSPD, Destinasi, tipologi",Bukan PII (nama kawasan dan tempat wisata publik)
2,"Koordinat (lat, lon)",Quasi-identifier RENDAH: titik tempat wisata p...
3,Uraian (dusun/desa/kecamatan),Quasi-identifier RENDAH: lokasi administratif ...


In [ ]:
# DATA ETHICS CHECKLIST 9 BUTIR (dengan bukti)
no_pii = len(pii_hit) == 0

checklist = pd.DataFrame([
 [1, "Sumber data sah & terdokumentasi", "Ya",
  f"Publisher: Dinas Pariwisata (Satu Data NTB); URL: {SOURCE_URL}; MD5 berkas: {MD5_AWAL}"],
 [2, "Lisensi dicek & penggunaan sesuai lisensi", "Ya, dicek (lisensi tidak ditentukan)",
  f"Halaman dataset: {LICENSE_TXT}. Karena tidak ada lisensi eksplisit, data dipakai terbatas untuk keperluan akademik (UTS) dengan menyebut sumber"],
  [3, "Ketentuan portal dicek", "Dicek, teks ketentuan tidak tersedia di portal",
  f"{LISENSI['syarat_portal']}"],
 [4, "Tujuan penggunaan jelas (purpose limitation)", "Ya",
  "Tujuan: tugas UTS Data Acquisition & Management; tidak untuk komersial atau profiling"],
 [5, "Tidak memuat PII langsung", "Ya" if no_pii else "CEK ULANG",
  f"Pemindaian regex email/telepon/NIK/medsos pada semua kolom teks: {len(pii_hit)} temuan"],
 [6, "Quasi-identifier dinilai & diminimalkan", "Ya",
  "Koordinat dan dusun/desa = titik wisata publik, bukan alamat individu; risiko re-identifikasi rendah"],
 [7, "Minimisasi data (hanya yang perlu)", "Ya",
  "API hanya meminta suhu dan curah hujan; scraping hanya satu tabel; tidak ada data personal dikumpulkan"],
 [8, "Akses web sopan & patuh aturan (robots.txt, rate limit, User-Agent)", "Ya (bukti lengkap di Soal 4)",
  f"timeout={TIMEOUT}s, jeda={DELAY}s, User-Agent berisi email kampus; tidak mengambil dari situs terlarang di soal"],
 [9, "Kredensial aman & integritas data terjaga", "Ya",
  "Tidak ada API key di notebook (Open-Meteo tanpa key); berkas raw read-only; MD5 tiap sumber dicatat di provenance"],
], columns=["No", "Butir", "Status", "Bukti"])

pd.set_option("display.max_colwidth", 130)
display(checklist)

checklist.to_csv(f"{META}/data_ethics_checklist.csv", index=False, encoding="utf-8")
print("Tersimpan:", f"{META}/data_ethics_checklist.csv")

,No,Butir,Status,Bukti
0,1,Sumber data sah & terdokumentasi,Ya,Publisher: Dinas Pariwisata (Satu Data NTB); URL: https://data.ntbprov.go.id/dataset/a2fbd8bd-6b85-4ce8-b743-a36c78e9f59d/show...
1,2,Lisensi dicek & penggunaan sesuai lisensi,"Ya, dicek (lisensi tidak ditentukan)",Halaman dataset: License Not Specified (tidak ditentukan di halaman dataset; level akses: Public). Karena tidak ada lisensi ek...
2,3,Ketentuan portal dicek,"Dicek, teks ketentuan tidak tersedia di portal","(""Tautan 'Ketentuan Pengguna' di footer portal sudah dicoba dibuka (15 Nov 2026)"", 'tetapi tidak menampilkan teks ketentuan (h..."
3,4,Tujuan penggunaan jelas (purpose limitation),Ya,Tujuan: tugas UTS Data Acquisition & Management; tidak untuk komersial atau profiling
4,5,Tidak memuat PII langsung,Ya,Pemindaian regex email/telepon/NIK/medsos pada semua kolom teks: 0 temuan
5,6,Quasi-identifier dinilai & diminimalkan,Ya,"Koordinat dan dusun/desa = titik wisata publik, bukan alamat individu; risiko re-identifikasi rendah"
6,7,Minimisasi data (hanya yang perlu),Ya,API hanya meminta suhu dan curah hujan; scraping hanya satu tabel; tidak ada data personal dikumpulkan
7,8,"Akses web sopan & patuh aturan (robots.txt, rate limit, User-Agent)",Ya (bukti lengkap di Soal 4),"timeout=20s, jeda=2.0s, User-Agent berisi email kampus; tidak mengambil dari situs terlarang di soal"
8,9,Kredensial aman & integritas data terjaga,Ya,Tidak ada API key di notebook (Open-Meteo tanpa key); berkas raw read-only; MD5 tiap sumber dicatat di provenance


Tersimpan: data/meta/data_ethics_checklist.csv


In [ ]:
# SOAL 3-5 LENGKAP: API + robots.txt + scraping + merge + simpan
# Prasyarat: sel 1-10 sudah dijalankan (df, RAW, PROVENANCE, add_provenance,
#            KAB_PATTERNS, LISENSI, checklist, dll.)

import re, io, json, time, glob, requests, zipfile
from urllib.robotparser import RobotFileParser
from urllib.parse import urlparse

# Bersihkan sisa run sebelumnya agar sel aman diulang
for f in glob.glob(f"{RAW}/openmeteo_*.json"):
    os.remove(f)                                                   # hapus JSON lama
PROVENANCE[:] = [p for p in PROVENANCE if p["sumber"].startswith("Dataset")]   # sisakan catatan CSV

# BARU: penjaga syarat_portal (harus satu teks, bukan tuple)
if isinstance(LISENSI["syarat_portal"], (tuple, list)):
    LISENSI["syarat_portal"] = " ".join(LISENSI["syarat_portal"])
checklist.loc[checklist["No"] == 3, "Bukti"] = LISENSI["syarat_portal"]

# 0. Fungsi GET sopan (timeout, User-Agent, jeda, retry)
SESSION = requests.Session()
SESSION.headers.update({"User-Agent": USER_AGENT})   # berisi email kampus
MAX_RETRY = 4

def polite_get(url, params=None):
    for percobaan in range(1, MAX_RETRY + 1):
        try:
            r = SESSION.get(url, params=params, timeout=TIMEOUT)
            print(f"[{percobaan}] status {r.status_code}  {r.url[:90]}")
            if r.status_code in (429, 500, 502, 503, 504):      # error sementara -> retry
                time.sleep(int(r.headers.get("Retry-After", 2 ** percobaan)))
                continue
            r.raise_for_status()                                # 4xx lain -> gagal
            time.sleep(DELAY)                                   # jeda antar-request
            return r
        except (requests.Timeout, requests.ConnectionError) as e:
            print(f"[{percobaan}] error jaringan: {e}")
            time.sleep(2 ** percobaan)                          # backoff
    raise RuntimeError(f"Gagal mengambil {url} setelah {MAX_RETRY} percobaan")

# SOAL 3: API Open-Meteo (tanpa API key -> tidak ada kredensial di notebook)
# Titik pusat (median) tiap KSPD: 11 titik, bukan 266 destinasi

centroid = (df.groupby("KSPD")
              .agg(lat=("lat", "median"), lon=("lon", "median"), n_destinasi=("Destinasi", "count"))
              .round(5).reset_index())
print("Jumlah titik KSPD:", len(centroid))

API_URL = "https://api.open-meteo.com/v1/forecast"
baris_api = []
for _, c in centroid.iterrows():
    params = {"latitude": c.lat, "longitude": c.lon,
              "daily": "temperature_2m_max,temperature_2m_min,precipitation_sum",
              "timezone": "Asia/Makassar", "forecast_days": 7}
    r = polite_get(API_URL, params=params)
    js = r.json()

    # mennyimpan respons mentah apa adanya di data/raw (nama berkas aman, tanpa spasi)
    nama_aman = re.sub(r"\W+", "_", c.KSPD)[:40]
    raw_path = f"{RAW}/openmeteo_{nama_aman}.json"
    with open(raw_path, "w", encoding="utf-8") as f:
        json.dump(js, f, ensure_ascii=False)
    add_provenance(f"Open-Meteo forecast - {c.KSPD}", r.url, raw_path,
                   "Open-Meteo: data CC BY 4.0; API gratis untuk non-komersial", "API publik (JSON)")

    # normalisasi JSON -> tabel (1 hari = 1 baris)
    t = pd.DataFrame(js["daily"]); t.insert(0, "KSPD", c.KSPD)
    t["lat_api"], t["lon_api"] = js["latitude"], js["longitude"]
    baris_api.append(t)

weather = pd.concat(baris_api, ignore_index=True).rename(columns={"time": "tanggal"})
print(f"\nSOAL 3 -> baris tabel API: {len(weather)} (minimal 30: {'OK' if len(weather) >= 30 else 'TIDAK'})")
display(weather.head(8))

# SOAL 4: cek robots.txt + scraping tabel statis Wikipedia

WIKI_URL = "https://id.wikipedia.org/wiki/Nusa_Tenggara_Barat"
base = f"{urlparse(WIKI_URL).scheme}://{urlparse(WIKI_URL).netloc}"

# 4a. Ambil & periksa robots.txt SEBELUM scraping
rob = polite_get(base + "/robots.txt")
robots_path = f"{RAW}/robots_id_wikipedia.txt"
open(robots_path, "w", encoding="utf-8").write(rob.text)
add_provenance("robots.txt id.wikipedia.org", base + "/robots.txt", robots_path, "Dokumen aturan situs", "HTTP GET")

rp = RobotFileParser(); rp.parse(rob.text.splitlines())
diizinkan = rp.can_fetch(USER_AGENT, WIKI_URL)
bukti_robots = {"waktu_cek": dt.datetime.now().isoformat(timespec="seconds"),
                "robots_url": base + "/robots.txt", "halaman_target": WIKI_URL,
                "user_agent": USER_AGENT, "diizinkan": diizinkan,
                "crawl_delay_robots": rp.crawl_delay(USER_AGENT), "jeda_dipakai_detik": DELAY}
json.dump(bukti_robots, open(f"{META}/bukti_robots_check.json", "w"), indent=2, ensure_ascii=False)
print("\nSOAL 4 -> bukti robots.txt:"); display(pd.Series(bukti_robots, name="nilai").to_frame())
assert diizinkan, "robots.txt melarang halaman ini -> JANGAN di-scrape."

# 4b. Ambil halaman, simpan HTML mentah, ekstrak tabel dengan pd.read_html
page = polite_get(WIKI_URL)
html_path = f"{RAW}/wikipedia_nusa_tenggara_barat.html"
open(html_path, "w", encoding="utf-8").write(page.text)
add_provenance("Wikipedia ID - Nusa Tenggara Barat", WIKI_URL, html_path, "CC BY-SA 4.0 (teks Wikipedia)",
               "web scraping halaman statis (pd.read_html)")
tables = pd.read_html(io.StringIO(page.text))
print("Jumlah tabel di halaman:", len(tables))

def norm_name(s):
    """'Kabupaten Bima' -> 'Bima'; 'Kota Bima' -> 'Kota Bima'; buang catatan kaki [1]."""
    s = re.sub(r"\[.*?\]", "", str(s)).strip()
    s = re.sub(r"^(Kabupaten|Kab\.)\s+", "", s, flags=re.I)
    return re.sub(r"^Kota\s+", "Kota ", s, flags=re.I)

VALID = {k for k, _ in KAB_PATTERNS}      # 10 kabupaten/kota (dari sel pecah Uraian)

def flat(cols):
    return [" ".join(map(str, c)).strip() if isinstance(c, tuple) else str(c) for c in cols]

# pilih tabel & kolom nama yang paling banyak cocok dengan 10 kabupaten/kota NTB
terbaik = (0, None, None)
for i, t in enumerate(tables):
    t = t.copy(); t.columns = flat(t.columns)
    for col in t.columns:
        hit = t[col].map(norm_name).isin(VALID).sum()
        if hit > terbaik[0]: terbaik = (hit, i, col)
print("Tabel terpilih (jumlah cocok, indeks, kolom):", terbaik)
assert terbaik[0] >= 8, "Tabel kabupaten tidak ketemu otomatis; cek variabel `tables` dan pilih indeks manual."

scrape = tables[terbaik[1]].copy(); scrape.columns = flat(scrape.columns)
scrape["kab_key"] = scrape[terbaik[2]].map(norm_name)
scrape = scrape[scrape["kab_key"].isin(VALID)].drop_duplicates("kab_key").reset_index(drop=True)
scrape = scrape.rename(columns={c: f"wiki_{c}" for c in scrape.columns if c != "kab_key"})
print(f"\nSOAL 4 -> baris hasil scraping: {len(scrape)}"); display(scrape)

# Perbarui butir 8 checklist dengan bukti nyata
checklist.loc[checklist["No"] == 8, "Status"] = "Ya (terbukti)"
checklist.loc[checklist["No"] == 8, "Bukti"] = (
    f"robots.txt id.wikipedia.org dicek ({bukti_robots['waktu_cek']}), can_fetch={diizinkan}; "
    f"timeout={TIMEOUT}s, jeda={DELAY}s, retry<={MAX_RETRY}; bukti: data/meta/bukti_robots_check.json")
checklist.to_csv(f"{META}/data_ethics_checklist.csv", index=False, encoding="utf-8")

# SOAL 5: merge (validate) + bukti tanpa duplikasi + simpan + metadata
# API -> 1 baris per KSPD (agar relasi many-to-one)

weather_kspd = (weather.groupby("KSPD").agg(
    suhu_max_rata2_C=("temperature_2m_max", "mean"), suhu_min_rata2_C=("temperature_2m_min", "mean"),
    hujan_total_7hari_mm=("precipitation_sum", "sum"),
    tgl_awal=("tanggal", "min"), tgl_akhir=("tanggal", "max")).round(2).reset_index())
assert weather_kspd["KSPD"].is_unique

n_awal = len(df)
m1 = df.merge(weather_kspd, on="KSPD", how="left", validate="many_to_one", indicator="_m_api")   # berkas x API
m2 = m1.merge(scrape, on="kab_key", how="left", validate="many_to_one", indicator="_m_wiki")     # x scraping

bukti_merge = {
    "baris_awal_berkas": n_awal, "baris_setelah_merge_API": len(m1), "baris_akhir": len(m2),
    "jumlah_sama_dengan_awal": len(m2) == n_awal,
    "kunci_#_unik": bool(m2["#"].is_unique),
    "nama_destinasi_unik": bool(m2["Destinasi"].is_unique),
    "baris_duplikat_penuh": int(m2.drop(columns=["_m_api", "_m_wiki"]).duplicated().sum()),
    "cocok_API": int((m2["_m_api"] == "both").sum()),
    "cocok_scraping": int((m2["_m_wiki"] == "both").sum()),
    "tidak_cocok_scraping (kab_key kosong)": int((m2["_m_wiki"] != "both").sum()),
}
assert bukti_merge["jumlah_sama_dengan_awal"] and bukti_merge["kunci_#_unik"] and bukti_merge["baris_duplikat_penuh"] == 0
print("\nSOAL 5 -> bukti tidak ada duplikasi:"); display(pd.Series(bukti_merge, name="nilai").to_frame())

final = m2.drop(columns=["_m_api", "_m_wiki"])
OUT_CSV = f"{PROC}/destinasi_ntb_merged.csv"
final.to_csv(OUT_CSV, index=False, encoding="utf-8")
weather.to_csv(f"{PROC}/openmeteo_ntb_normalized.csv", index=False, encoding="utf-8")
scrape.to_csv(f"{PROC}/wikipedia_kabupaten_ntb.csv", index=False, encoding="utf-8")

# bukti berkas mentah tidak berubah
assert md5_file(RAW_CSV) == MD5_AWAL, "Berkas mentah berubah!"
print("MD5 CSV mentah tidak berubah:", MD5_AWAL)

# metadata hasil gabungan
metadata = {
    "judul": "Destinasi Wisata NTB per KSPD + Cuaca (Open-Meteo) + Atribut Kabupaten (Wikipedia)",
    "pembuat": "Era (NIM 2501010007) - UTS Data Acquisition & Management, INSTIKI",
    "dibuat": dt.datetime.now().isoformat(timespec="seconds"),
    "baris": int(final.shape[0]), "kolom": int(final.shape[1]), "nama_kolom": final.columns.tolist(),
    "kunci_merge": {"berkas-API": "KSPD (many_to_one)", "hasil-scraping": "kab_key (many_to_one)"},
    "bukti_merge": bukti_merge, "md5_hasil": md5_file(OUT_CSV),
    "catatan": ("kab_key diturunkan dari teks Lokasi; baris tanpa nama kabupaten dibiarkan kosong. "
                "Cuaca = prakiraan 7 hari, bukan data historis."),
}
json.dump(metadata, open(f"{META}/metadata_merged.json", "w"), indent=2, ensure_ascii=False)

# provenance semua sumber
prov = pd.DataFrame(PROVENANCE)
prov.to_csv(f"{META}/provenance.csv", index=False, encoding="utf-8")
json.dump(PROVENANCE, open(f"{META}/provenance.json", "w"), indent=2, ensure_ascii=False)
print(f"\nPROVENANCE: {len(prov)} sumber tercatat"); display(prov[["sumber", "tanggal_akses", "lisensi", "md5"]])

# ringkasan storage seluruh sumber + paket pengumpulan
total = sum(p["ukuran_bytes"] for p in PROVENANCE)
print(f"Total raw semua sumber: {total/1024:.1f} KB -> estimasi 1 tahun (x12, +30%): {total*12*1.3/1024**2:.2f} MB")
with zipfile.ZipFile("paket_uts_dam_2501010007.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for root, _, fs in os.walk("data"):
        for f in fs: z.write(os.path.join(root, f))
print("Paket dibuat: paket_uts_dam_2501010007.zip")

# ringkasan pemeriksaan akhir
checklist.to_csv(f"{META}/data_ethics_checklist.csv", index=False, encoding="utf-8")
print("\n===== RINGKASAN PEMERIKSAAN =====")
print("User-Agent       :", USER_AGENT)
print("Jumlah provenance:", len(PROVENANCE), "(target 14)")
pd.set_option("display.max_colwidth", 250)
display(checklist.loc[checklist["No"].isin([3, 8]), ["No", "Status", "Bukti"]])
display(final.head())

Jumlah titik KSPD: 11
[1] status 200  https://api.open-meteo.com/v1/forecast?latitude=-8.47709&longitude=117.07667&daily=tempera
[1] status 200  https://api.open-meteo.com/v1/forecast?latitude=-8.69166&longitude=117.05587&daily=tempera
[1] status 200  https://api.open-meteo.com/v1/forecast?latitude=-8.6831&longitude=118.40841&daily=temperat
[1] status 200  https://api.open-meteo.com/v1/forecast?latitude=-8.84355&longitude=116.26526&daily=tempera
[1] status 200  https://api.open-meteo.com/v1/forecast?latitude=-8.58422&longitude=116.10181&daily=tempera
[1] status 200  https://api.open-meteo.com/v1/forecast?latitude=-8.69015&longitude=116.81025&daily=tempera
[1] status 200  https://api.open-meteo.com/v1/forecast?latitude=-8.43974&longitude=116.48398&daily=tempera
[1] status 200  https://api.open-meteo.com/v1/forecast?latitude=-8.55463&longitude=117.63507&daily=tempera
[1] status 200  https://api.open-meteo.com/v1/forecast?latitude=-8.35645&longitude=116.24166&daily=tempera
[1] status 200 

,KSPD,tanggal,temperature_2m_max,temperature_2m_min,precipitation_sum,lat_api,lon_api
0,KSPD ALAS UTAN DAN SEKITARNYA,2026-11-16,31.9,25.6,9.8,-8.471002,117.061850
1,KSPD ALAS UTAN DAN SEKITARNYA,2026-11-17,30.8,24.7,5.2,-8.471002,117.061850
2,KSPD ALAS UTAN DAN SEKITARNYA,2026-11-18,32.0,25.0,4.8,-8.471002,117.061850
3,KSPD ALAS UTAN DAN SEKITARNYA,2026-11-19,30.7,24.9,3.9,-8.471002,117.061850
4,KSPD ALAS UTAN DAN SEKITARNYA,2026-11-20,32.0,25.1,11.4,-8.471002,117.061850
5,KSPD ALAS UTAN DAN SEKITARNYA,2026-11-21,31.0,24.9,23.1,-8.471002,117.061850
6,KSPD ALAS UTAN DAN SEKITARNYA,2026-11-22,29.8,24.9,13.2,-8.471002,117.061850
7,KSPD BATU HIJAU DODORINTI DAN SEKITARNYA,2026-11-16,32.4,20.8,11.1,-8.681898,116.976746


[1] status 200  https://id.wikipedia.org/robots.txt

SOAL 4 -> bukti robots.txt:


,nilai
waktu_cek,2026-11-16T02:52:25
robots_url,https://id.wikipedia.org/robots.txt
halaman_target,https://id.wikipedia.org/wiki/Nusa_Tenggara_Barat
user_agent,INSTIKI-UTS-DAM/1.0 (mahasiswa 2501010007; kontak: info@instiki.ac.id)
diizinkan,True
crawl_delay_robots,None
jeda_dipakai_detik,2.0


[1] status 200  https://id.wikipedia.org/wiki/Nusa_Tenggara_Barat
Jumlah tabel di halaman: 12
Tabel terpilih (jumlah cocok, indeks, kolom): (np.int64(10), 4, 'Kabupaten/kota')

SOAL 4 -> baris hasil scraping: 10


,wiki_No.,wiki_Kabupaten/kota,wiki_Ibu kota,wiki_Bupati/wali kota,wiki_Luas wilayah (km2)[23],wiki_Jumlah penduduk (2025)[23],wiki_Kecamatan,wiki_Kelurahan/desa,wiki_Lambang alt,wiki_Peta lokasi,kab_key
0,1,Kabupaten Bima,Woha,Ady Mahyudi,"4.208,861",541.582,18,-/191,NaN,NaN,Bima
1,2,Kabupaten Dompu,Dompu,Bambang Firdaus,"2.270,360",277.837,8,9/72,NaN,NaN,Dompu
2,3,Kabupaten Lombok Barat,Gerung,Lalu Ahmad Zaini,919440,762.757,10,3/119,NaN,NaN,Lombok Barat
3,4,Kabupaten Lombok Tengah,Praya,Lalu Pathul Bahri,"1.169,029",1.128.716,12,12/127,NaN,NaN,Lombok Tengah
4,5,Kabupaten Lombok Timur,Selong,Haerul Warisin,"1.607,377",1.457.591,20,15/239,NaN,NaN,Lombok Timur
5,6,Kabupaten Lombok Utara,Tanjung,Najmul Akhyar,810185,269.989,5,-/43,NaN,NaN,Lombok Utara
6,7,Kabupaten Sumbawa,Sumbawa,Syarafuddin Jarot,"6.645,723",527.715,24,8/157,NaN,NaN,Sumbawa
7,8,Kabupaten Sumbawa Barat,Taliwang,Amar Nurmansyah,"1.739,564",156.059,8,7/57,NaN,NaN,Sumbawa Barat
8,9,Kota Bima,-,A. Rahman H. Abidin,208389,165.113,5,38/-,NaN,NaN,Kota Bima
9,10,Kota Mataram,-,Mohan Roliskana,60063,461.936,6,50/-,NaN,NaN,Kota Mataram


... (3 outputs left out)

,sumber,tanggal_akses,lisensi,md5
0,Dataset destinasi wisata KSPD NTB (CSV),2026-11-16T02:27:50,License Not Specified (tidak ditentukan di halaman dataset; level akses: Public),358dd8b2be8b073b32849f85da7f632a
1,Open-Meteo forecast - KSPD ALAS UTAN DAN SEKITARNYA,2026-11-16T02:52:03,Open-Meteo: data CC BY 4.0; API gratis untuk non-komersial,11bce16e74bd12508e442841207e50a8
2,Open-Meteo forecast - KSPD BATU HIJAU DODORINTI DAN SEKITARNYA,2026-11-16T02:52:05,Open-Meteo: data CC BY 4.0; API gratis untuk non-komersial,aa86bc9afb467663bc6dfcaea7a71b67
3,Open-Meteo forecast - KSPD HU’U DAN SEKITARNYA,2026-11-16T02:52:07,Open-Meteo: data CC BY 4.0; API gratis untuk non-komersial,b9808a05ecdffe31ff3ce78a59380051
4,Open-Meteo forecast - KSPD KUTA - MANDALIKA DAN SEKITARNYA,2026-11-16T02:52:09,Open-Meteo: data CC BY 4.0; API gratis untuk non-komersial,feaf789a7844d386648b5788e73cd356
5,Open-Meteo forecast - KSPD MATARAM METRO DAN SEKITARNYA,2026-11-16T02:52:11,Open-Meteo: data CC BY 4.0; API gratis untuk non-komersial,4fae9a1b2a8e028df5333f24816cb622
6,Open-Meteo forecast - KSPD POTOTANO - MALUK DAN SEKITARNYA,2026-11-16T02:52:13,Open-Meteo: data CC BY 4.0; API gratis untuk non-komersial,a1c30180a2056c1af4c7755c62948d67
7,Open-Meteo forecast - KSPD RASIMAS - SEMBALUN DAN SEKITARNYA,2026-11-16T02:52:15,Open-Meteo: data CC BY 4.0; API gratis untuk non-komersial,433f6ac45d28360f610c7f1da6704b80
8,Open-Meteo forecast - KSPD SAMOTA (TELUK SALEH - MOYO – TAMBORA) DAN SEKITARNYA,2026-11-16T02:52:17,Open-Meteo: data CC BY 4.0; API gratis untuk non-komersial,5f356d084f5134eb662c63136b6023d3
9,"Open-Meteo forecast - KSPD SENGGIGI, 3 GILI DAN SEKITARNYA",2026-11-16T02:52:19,Open-Meteo: data CC BY 4.0; API gratis untuk non-komersial,76c81f345769797052a556332cec45e1


Total raw semua sumber: 545.7 KB -> estimasi 1 tahun (x12, +30%): 8.31 MB
Paket dibuat: paket_uts_dam_2501010007.zip

===== RINGKASAN PEMERIKSAAN =====
User-Agent       : INSTIKI-UTS-DAM/1.0 (mahasiswa 2501010007; kontak: info@instiki.ac.id)
Jumlah provenance: 14 (target 14)


,No,Status,Bukti
2,3,"Dicek, teks ketentuan tidak tersedia di portal","Tautan 'Ketentuan Pengguna' di footer portal sudah dicoba dibuka (15 Nov 2026) tetapi tidak menampilkan teks ketentuan (hanya muncul pop-up survei kepuasan). Halaman dataset: lisensi 'License Not Specified', level akses Public. Data dipakai han..."
7,8,Ya (terbukti),"robots.txt id.wikipedia.org dicek (2026-11-16T02:52:25), can_fetch=True; timeout=20s, jeda=2.0s, retry<=4; bukti: data/meta/bukti_robots_check.json"


,#,Kode Provinsi,Nama Provinsi,KSPD,Destinasi,Uraian,Koordinat,Tahun,lat,lon,...,wiki_No.,wiki_Kabupaten/kota,wiki_Ibu kota,wiki_Bupati/wali kota,wiki_Luas wilayah (km2)[23],wiki_Jumlah penduduk (2025)[23],wiki_Kecamatan,wiki_Kelurahan/desa,wiki_Lambang alt,wiki_Peta lokasi
0,1,52,Nusa Tenggara Barat,KSPD WAWORADA - SAPE DAN SEKITARNYA,Sigi Kamina/kalo Du,"• Tipologi : Wisata Ziarah-religi • Lokasi : Dusun Kamina, Desa Kalodu, Kec. Langgudu, Kab. Bima • Aksesibilitas : Kota Bima-karumbu (± 54,2 Km Atau Sekitar 1 Jam 30 Menit)","-8.665510, 118.845924",2025,-8.665510,118.845924,...,1.0,Kabupaten Bima,Woha,Ady Mahyudi,"4.208,861",541.582,18.0,-/191,NaN,NaN
1,2,52,Nusa Tenggara Barat,KSPD WAWORADA - SAPE DAN SEKITARNYA,Kompleks Rumah Adat Wawo,"• Tipologi : Wisata Budaya • Lokasi : Desa Maria Utara, Kec. Wawo, Kab. Bima • Aksesibilitas : Kota Bima-jalur Menuju Wawo (± 45 Menit)","-8.524548, 118.846067",2025,-8.524548,118.846067,...,1.0,Kabupaten Bima,Woha,Ady Mahyudi,"4.208,861",541.582,18.0,-/191,NaN,NaN
2,3,52,Nusa Tenggara Barat,KSPD WAWORADA - SAPE DAN SEKITARNYA,Desa Adat Sambori,"• Tipologi : Wisata Alam Dan Wisata Budaya • Lokasi : Desa Sambori, Kec. Lambitu, Kab. Bima • Aksesibilitas : Ibukota Kab. Bima-kec. Lambitu","-8.606385, 118.778023",2025,-8.606385,118.778023,...,1.0,Kabupaten Bima,Woha,Ady Mahyudi,"4.208,861",541.582,18.0,-/191,NaN,NaN
3,4,52,Nusa Tenggara Barat,KSPD WAWORADA - SAPE DAN SEKITARNYA,Wisata Alam Kalate Ndira,"• Tipologi : Wisata Alam • Lokasi : Dusun ‘diha, Desa Sie, Kec. Monta, Kab. Bima • Aksesibilitas : Kota Bima-desa Sie (± 1 Jam) Atau Desa Sihe-dusun Diha","-8.596853, 118.663070",2025,-8.596853,118.663070,...,1.0,Kabupaten Bima,Woha,Ady Mahyudi,"4.208,861",541.582,18.0,-/191,NaN,NaN
4,5,52,Nusa Tenggara Barat,KSPD WAWORADA - SAPE DAN SEKITARNYA,Air Terjun Tarlawi Wawo,"• Tipologi : Wisata Alam • Lokasi : Desa Tarlawi, Kec. Wawo, Kab. Bima • Aksesibilitas : Kota Bima-wawo (± 1,5 Jam)","-8.606373, 118.824722",2025,-8.606373,118.824722,...,1.0,Kabupaten Bima,Woha,Ady Mahyudi,"4.208,861",541.582,18.0,-/191,NaN,NaN
